# 02. Подготовка данных

Что делает ноутбук (все решения следуют из EDA, `01_eda.ipynb`):

1. Разбирает параметры объявлений: «Вид услуги», «Тип услуги», названия услуг из прайс-листа.
2. Собирает для каждого объявления текстовые поля для поиска и чистит числовые признаки (цена `-1`, битые координаты).
3. Строит **собственную валидацию**: откладывает ~6000 поисков из train, похожих на бенчмарк
   (62.5% текстов, которых «нет в train», и одна группа на текст).
4. Собирает **валидационный корпус** = корпус бенчмарка + объявления, выбранные в отложенных поисках.
   Так отложенные поиски решаются в тех же условиях, что и бенчмарк: нужное объявление спрятано среди ~190 тыс. чужих.
5. Сохраняет пары «запрос → объявление» для обучения моделей (без отложенных поисков — иначе утечка).

Результат — файлы в `artifacts/`. Их же загружаем на Kaggle для обучения нейросети.

In [1]:
import sys, time, json
from pathlib import Path
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc

sys.path.insert(0, '..')
from src.prep import (read_parquet, norm_text, ParamsParser, build_items, build_queries,
                      make_holdout, GROUP_KEY)

DATA = Path('../data')
ART = Path('../artifacts'); ART.mkdir(exist_ok=True)
SEED = 42

## 1. Загрузка

Train без описаний занимает ~1 ГБ памяти. Описания нужны только для уникальных объявлений,
поэтому читаем их отдельно и сразу обрезаем до 1500 символов (EDA: медиана длины ~1000).

In [2]:
t0 = time.time()
train_cols = [c for c in pq.ParquetFile(DATA / 'train.parquet').schema_arrow.names if c != 'item_description_raw']
train = read_parquet(DATA / 'train.parquet', columns=train_cols)
train['q_norm'] = train['search_query'].map(norm_text)

bench_items = read_parquet(DATA / 'benchmark_items.parquet')
bench_queries = read_parquet(DATA / 'benchmark_queries.parquet')
print(f'train {train.shape}, items {bench_items.shape}, queries {bench_queries.shape}, {time.time() - t0:.0f} c')

train (497673, 19), items (189212, 14), queries (2452, 6), 5 c


In [3]:
# Уникальные объявления train + их описания (обрезанные; длину полного описания сохраняем отдельно) — нужны для валидационного корпуса и пар для обучения
desc_tbl = pq.read_table(DATA / 'train.parquet', columns=['item_id', 'item_description_raw'])
desc_tbl = desc_tbl.append_column('desc_full_len', pc.utf8_length(desc_tbl['item_description_raw']))
desc_tbl = desc_tbl.set_column(1, 'item_description_raw', pc.utf8_slice_codeunits(desc_tbl['item_description_raw'], 0, 1500))
desc = desc_tbl.to_pandas().drop_duplicates('item_id')
del desc_tbl

item_cols = [c for c in bench_items.columns if c != 'item_description_raw']
train_items_raw = train[item_cols].drop_duplicates('item_id').merge(desc, on='item_id', how='left')
del desc
print('Уникальных объявлений в train:', len(train_items_raw))

Уникальных объявлений в train: 344825


## 2. Разбор параметров

Параметры — одна склеенная строка без разделителей («Вид услуги Ремонт и отделка Тип услуги Сантехника Место оказания услуг ...»).
Поэтому значения ищем по словарю известных значений, который строится по данным (см. `src/prep.py`).

In [4]:
param_texts = pd.concat([
    bench_items['item_infm_params_text'], train_items_raw['item_infm_params_text'],
    train['search_infm_params_text'].drop_duplicates(), bench_queries['search_infm_params_text'],
]).dropna()
parser = ParamsParser(param_texts)
print({k: len(v) for k, v in parser.vocab.items()})
print('Примеры «Вид услуги»:', parser.vocab['vid'][:10])
print('Примеры «Тип услуги»:', parser.vocab['type'][:10])

{'vid': 26, 'type': 102, 'auto': 12}
Примеры «Вид услуги»: ['Ремонт и обслуживание техники', 'Оборудование, производство', 'Монтаж и установка техники', 'Грузчики, складские услуги', 'Вывоз мусора и вторсырья', 'Доставка еды и продуктов', 'Праздники, мероприятия', 'Пассажирские перевозки', 'Сад, благоустройство', 'Охрана, безопасность']
Примеры «Тип услуги»: ['Озеленение, уход за садом и огородом', 'Отделка деревянных домов, бань, саун', 'Строительство гаражей, бань, веранд', 'Сварка, ковка, металлоконструкции', 'Поклейка обоев и малярные работы', 'Холодильники, морозильные камеры', 'Ремонт квартир и домов под ключ', 'Газовые котлы, водонагреватели', 'Фундаментные и бетонные работы', 'Варочные панели, духовые шкафы']


## 3. Таблицы объявлений

`build_items` делает поля для поиска (`title`, `struct`, `params`, `desc`, `dense_text`) и чистит числа.

In [5]:
t0 = time.time()
items_b = build_items(bench_items, parser); items_b['in_bench'] = 1
items_t = build_items(train_items_raw, parser); items_t['in_bench'] = items_t['item_id'].isin(set(items_b['item_id'])).astype(int)
print(f'{time.time() - t0:.0f} c')
print('Пустой «Вид услуги»:', (items_b['vid'] == '').mean().round(3), '| пустой «Тип услуги»:', (items_b['type'] == '').mean().round(3))
items_b[['title', 'struct', 'vid', 'type']].sample(5, random_state=1)

22 c
Пустой «Вид услуги»: 0.064 | пустой «Тип услуги»: 0.269


,title,struct,vid,type
121061,"Оценка авто для нотариуса, наследства Уварово",Деловые услуги ; Юридические услуги,Деловые услуги,Юридические услуги
183730,Доставка чернозема высший сорт,"Сад, благоустройство ; Земляные работы ; ; Ва...","Сад, благоустройство",Земляные работы
80108,"Вязаные игрушки ручной работы,крючком и спицами",Искусство,Искусство,
86414,Грузоперевозки Газель next с грузчиками,Грузоперевозки ; По городу,Грузоперевозки,По городу
157979,"Контент-съемка, мобильный фотограф, визуал, reels",Фото- и видеосъёмка,Фото- и видеосъёмка,


## 4. Валидация

Логика отбора (`make_holdout`):

* **группа** = один поиск: текст + локация + фильтры. Это аналог одной строки `benchmark_queries`;
* позитивы группы = все объявления, выбранные в этом поиске;
* 62.5% групп берём из редких текстов и удаляем из обучения **все** строки с этим текстом — так имитируем новые запросы;
* остальные группы подбираем так, чтобы частота их текста в оставшемся train совпадала с бенчмарком.

In [6]:
holdout, fit_mask = make_holdout(train, n_groups=6000, seed=SEED)
print('Отложено групп:', len(holdout), '| строк train для обучения:', fit_mask.sum(), 'из', len(train))
print(holdout['bucket'].value_counts(normalize=True).round(3))
print('Позитивов на группу:', holdout['item_ids'].str.len().describe().round(2).to_dict())

# Проверка на утечку: «unseen»-тексты не должны встречаться в обучающей части
fit_texts = set(train.loc[fit_mask, 'q_norm'])
print('Утечка unseen-текстов:', holdout.loc[holdout['bucket'] == 'unseen', 'q_norm'].isin(fit_texts).sum())

Отложено групп: 6000 | строк train для обучения: 489956 из 497673
bucket
unseen    0.625
2-5       0.147
1         0.096
6-20      0.090
>20       0.042
Name: proportion, dtype: float64
Позитивов на группу: {'count': 6000.0, 'mean': 1.06, 'std': 0.29, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 1.0, 'max': 8.0}


Утечка unseen-текстов: 0


## 5. Корпус

Корпус = объявления бенчмарка + позитивы отложенных поисков. Флаг `in_bench` отмечает объявления,
которые разрешено отдавать в `answer.csv`.

In [7]:
val_pos = set(i for ids in holdout['item_ids'] for i in ids)
extra = items_t[items_t['item_id'].isin(val_pos) & (items_t['in_bench'] == 0)]
corpus = pd.concat([items_b, extra], ignore_index=True)
assert corpus['item_id'].is_unique
print(f'Корпус: {len(corpus)} объявлений ({len(extra)} добавлено из валидации)')
print('Доля позитивов валидации, которые уже есть в корпусе бенчмарка:', round(len(val_pos & set(items_b['item_id'])) / len(val_pos), 3))
corpus.to_parquet(ART / 'corpus.parquet', index=False)

Корпус: 195072 объявлений (5860 добавлено из валидации)


Доля позитивов валидации, которые уже есть в корпусе бенчмарка: 0.072


## 6. Запросы: валидационные и бенчмарк

In [8]:
q_bench = build_queries(bench_queries, parser)
q_val = build_queries(holdout.drop(columns=['q_norm']).rename(columns={'search_query': 'search_query'}), parser)
q_val['relevant'] = holdout['item_ids'].values
q_val['bucket'] = holdout['bucket'].values

q_cols = ['query_id', 'search_query', 'q_norm', 'search_location_id', 'search_infm_params_text', 'search_category',
          'f_vid', 'f_type', 'f_auto', 'f_rating', 'dense_query']
q_bench[q_cols].to_parquet(ART / 'queries_bench.parquet', index=False)
q_val[q_cols + ['relevant', 'bucket']].to_parquet(ART / 'queries_val.parquet', index=False)
q_val[['search_query', 'dense_query', 'relevant']].head()

,search_query,dense_query,relevant
0,обучение накс,обучение накс,[e5c4b5a327d0875d]
1,бывшие пожалеет,бывшие пожалеет,[91c245b18a849b48]
2,аренда сутки электромопед,"аренда сутки электромопед | Оборудование, прои...",[7c98d47a4bb4dd2b]
3,мото перевозка,мото перевозка,[e0f41d2dc7bdf3a2]
4,утепление крыши пенополиуретаном,утепление крыши пенополиуретаном,[a5599b2980c6ba29]


## 7. Обучающие пары

Из строк train, не попавших в валидацию:

* `train_fit.parquet` — пары для статистических моделей (гео-приор, классификатор микрокатегории);
* `dense_pairs.parquet` — пары «текст запроса → текст объявления» для дообучения нейросети.
  Дубли убираем, число пар на один текст ограничиваем 20: иначе «маникюр» с сотнями выборов
  задавит редкие запросы, а бенчмарк состоит в основном из редких.

In [9]:
fit = train.loc[fit_mask].copy()
fit_q = build_queries(fit[['search_query', 'search_infm_params_text']].drop_duplicates(), parser)
fit = fit.merge(fit_q[['search_query', 'search_infm_params_text', 'f_vid', 'f_type', 'f_auto', 'dense_query']],
                on=['search_query', 'search_infm_params_text'], how='left')
fit_cols = ['q_norm', 'search_location_id', 'search_infm_params_text', 'f_vid', 'f_type', 'dense_query',
            'item_id', 'item_location_id', 'item_microcat_id', 'item_latitude', 'item_longitude', 'search_is_delivery_search']
fit[fit_cols].to_parquet(ART / 'train_fit.parquet', index=False)
print('train_fit:', fit.shape)

train_fit: (489956, 23)


In [10]:
pairs = fit[['q_norm', 'dense_query', 'item_id']].drop_duplicates(['q_norm', 'item_id'])
pairs = pairs.sample(frac=1.0, random_state=SEED).groupby('q_norm', sort=False).head(20)
pairs = pairs.merge(items_t[['item_id', 'dense_text']], on='item_id', how='left')
pairs = pairs.rename(columns={'dense_query': 'query', 'dense_text': 'doc'})[['q_norm', 'query', 'doc', 'item_id']]
pairs.to_parquet(ART / 'dense_pairs.parquet', index=False)
print('Пар для нейросети:', len(pairs), '| уникальных текстов запросов:', pairs['q_norm'].nunique())
pairs.sample(5, random_state=0)[['query', 'doc']]

Пар для нейросети: 214107 | уникальных текстов запросов: 69956


,query,doc
163943,"lpg массаж лица | Красота, здоровье СПА-услуги...","LPG массаж на аппарате LovePG. Красота, здоров..."
54935,прием лома | Вывоз мусора и вторсырья,Прием и вывоз металлолома. Вывоз мусора и втор...
41849,изготовление металлических печатей,"Печать каталогов, брошюр, журналов. , . Катал..."
161267,монтаж потолка армстронг,"Монтаж подвесных потолков Армстронг, Грильято...."
61891,мытье окон с улицы | Уборка Генеральная уборка,"Клининг. Уборка квартир, домов и офисов. Уборк..."


In [11]:
for f in sorted(ART.glob('*.parquet')):
    print(f'{f.name:25s} {f.stat().st_size / 2**20:7.1f} MB')

corpus.parquet              352.9 MB
dense_pairs.parquet         104.3 MB
queries_bench.parquet         0.3 MB
queries_val.parquet           0.7 MB
train_fit.parquet            36.4 MB
